# Kapanin — EDA untuk keputusan stok dan diskon

**Pertanyaan bisnis:** produk mana yang stoknya berisiko tersisa, kapan perlu ditinjau, dan berapa ruang margin untuk tindakan harga? Notebook ini menghasilkan ringkasan penjualan, pola permintaan, perilaku diskon, serta tabel prioritas produk. Satu toko; tanggal acuan stok mengikuti `stock_as_of_date`.

Data ini adalah prototipe sintetis. Angka di bawah berguna untuk menguji alur analisis, bukan bukti hasil bisnis di toko nyata. Notebook ini belum mengestimasi efek kausal diskon dan belum menentukan persentase diskon optimal.


## 0. Muat data

Di repo lokal, notebook mencari `data/store_1`. Di Colab, notebook membaca CSV dari branch `main` GitHub. Pastikan ketiga file sudah tersedia dan repo dapat diakses publik. Jalankan seluruh sel dari atas ke bawah. Kebutuhan inti: Pandas dan NumPy; Matplotlib hanya untuk grafik.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

candidates = [
    Path.cwd() / "data" / "store_1",
    Path.cwd() / "Kapanin" / "data" / "store_1",
    Path.cwd().parent / "data" / "store_1",
]
required_files = ("stores.csv", "products.csv", "transactions.csv")
DATA_DIR = next((path for path in candidates if all((path / name).exists() for name in required_files)), None)
RAW_BASE_URL = "https://raw.githubusercontent.com/faizwibawa/Kapanin/main/data/store_1"

def csv_source(filename):
    return DATA_DIR / filename if DATA_DIR is not None else f"{RAW_BASE_URL}/{filename}"

print("Sumber data:", DATA_DIR.resolve() if DATA_DIR is not None else RAW_BASE_URL)

try:
    import matplotlib.pyplot as plt
    HAS_PLOTS = True
    plt.rcParams.update({
        "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
        "axes.titlesize": 11, "axes.titleweight": "bold", "font.size": 9,
    })
except ImportError:
    HAS_PLOTS = False
    print("Matplotlib tidak tersedia. Di Colab grafik akan tampil setelah Matplotlib tersedia.")


In [ ]:
stores = pd.read_csv(csv_source("stores.csv"), dtype={"store_id": "string"})
products = pd.read_csv(
    csv_source("products.csv"),
    dtype={"product_id": "string", "store_id": "string", "sku": "string"},
    parse_dates=["stock_as_of_date", "stock_received_date", "expiry_date"],
)
transactions = pd.read_csv(
    csv_source("transactions.csv"),
    dtype={"transaction_id": "string", "store_id": "string", "product_id": "string"},
    parse_dates=["transaction_date"],
)
snapshot = products.stock_as_of_date.max()
print(f"Toko: {stores.store_name.iloc[0]} | snapshot stok: {snapshot.date()}")
print(f"Produk: {len(products):,} | transaksi: {len(transactions):,}")
print(f"Periode transaksi: {transactions.transaction_date.min().date()} s.d. {transactions.transaction_date.max().date()}")


## 1. Pemeriksaan kualitas data

Pemeriksaan ini memisahkan masalah data yang dapat mengubah keputusan: kunci ganda, transaksi tanpa produk, nilai harga tidak masuk akal, tanggal transaksi setelah snapshot, dan stok yang tanggal kedaluwarsanya sudah tiba. Jumlah temuan ditampilkan; analisis sebaiknya ditahan bila ada masalah pada kunci atau harga.


In [ ]:
quality_checks = {
    "duplikat_store_id": stores.store_id.duplicated().sum(),
    "duplikat_product_id": products.product_id.duplicated().sum(),
    "duplikat_transaction_id": transactions.transaction_id.duplicated().sum(),
    "transaksi_tanpa_produk": (~transactions.product_id.isin(products.product_id)).sum(),
    "produk_tanpa_toko": (~products.store_id.isin(stores.store_id)).sum(),
    "transaksi_tanpa_toko": (~transactions.store_id.isin(stores.store_id)).sum(),
    "stok_negatif": products.stock_on_hand.lt(0).sum(),
    "harga_atau_biaya_tidak_positif": (products.list_price.le(0) | products.unit_cost.le(0)).sum(),
    "harga_jual_tidak_positif": transactions.unit_sale_price.le(0).sum(),
    "diskon_di_luar_0_sampai_1": (~transactions.discount_rate.between(0, 1)).sum(),
    "transaksi_setelah_snapshot": transactions.transaction_date.ge(snapshot).sum(),
    "stok_positif_kedaluwarsa_hari_ini_atau_lalu": ((products.stock_on_hand > 0) & (products.expiry_date <= snapshot)).sum(),
}
quality_report = pd.Series(quality_checks, name="jumlah_temuan").to_frame()
print(quality_report.to_string())
assert quality_report.loc[
    ["duplikat_store_id", "duplikat_product_id", "duplikat_transaction_id", "transaksi_tanpa_produk",
     "produk_tanpa_toko", "transaksi_tanpa_toko", "stok_negatif", "harga_atau_biaya_tidak_positif",
     "harga_jual_tidak_positif", "diskon_di_luar_0_sampai_1", "transaksi_setelah_snapshot"],
    "jumlah_temuan"
].eq(0).all(), "Ada masalah data inti; periksa quality_report sebelum melanjutkan."


## 2. Kinerja penjualan

Ukur unit dan omzet berdasarkan **harga yang benar-benar dibayar**. Lihat tren mingguan serta kategori dengan kontribusi terbesar. Tren historis membantu memahami skala aktivitas, tetapi tidak boleh langsung ditafsirkan sebagai efek dari diskon atau stok.


In [ ]:
sales = transactions.merge(
    products[["product_id", "category", "unit_cost"]], on="product_id", how="left", validate="many_to_one"
)
sales["revenue"] = sales.quantity * sales.unit_sale_price
sales["week_start"] = sales.transaction_date - pd.to_timedelta(sales.transaction_date.dt.weekday, unit="D")
weekly_sales = sales.groupby("week_start", as_index=False).agg(
    units_sold=("quantity", "sum"), revenue=("revenue", "sum"), transactions=("transaction_id", "count")
)
# Hanya minggu Senin–Minggu penuh; minggu pertama/terakhir bisa parsial.
first_full_monday = transactions.transaction_date.min() + pd.Timedelta(
    days=(-transactions.transaction_date.min().weekday()) % 7
)
last_date = transactions.transaction_date.max()
weekly_sales = weekly_sales.loc[
    (weekly_sales.week_start >= first_full_monday) &
    (weekly_sales.week_start + pd.Timedelta(days=6) <= last_date)
].reset_index(drop=True)
category_sales = sales.groupby("category", as_index=False).agg(
    units_sold=("quantity", "sum"), revenue=("revenue", "sum"), products_sold=("product_id", "nunique")
).sort_values("revenue", ascending=False)
category_sales["revenue_share"] = category_sales.revenue / category_sales.revenue.sum()
print("Penjualan mingguan terakhir:")
print(weekly_sales.tail(8).to_string(index=False))
print("\n10 kategori dengan omzet tertinggi:")
print(category_sales.head(10).to_string(index=False, formatters={"revenue_share": "{:.1%}".format}))


In [ ]:
# Grafik 1–3: tren unit, omzet, dan kontribusi kategori.
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    weekly_view = weekly_sales.copy()
    weekly_view["avg_4w"] = weekly_view.units_sold.rolling(4, min_periods=4).mean()
    axes[0].plot(weekly_view.week_start, weekly_view.units_sold, color="#9ecae1", linewidth=1, label="Mingguan")
    axes[0].plot(weekly_view.week_start, weekly_view.avg_4w, color="#0b6e75", linewidth=2, label="Rata-rata 4 minggu")
    axes[0].set(title="Unit terjual per minggu penuh", ylabel="Unit")
    axes[0].legend(frameon=False)
    axes[1].plot(weekly_view.week_start, weekly_view.revenue / 1_000_000, color="#e07a3f")
    axes[1].set(title="Omzet per minggu penuh", ylabel="Juta rupiah")
    top_categories = category_sales.head(10).sort_values("revenue")
    axes[2].barh(top_categories.category, top_categories.revenue / 1_000_000, color="#367c8d")
    axes[2].set(title="10 kategori dengan omzet tertinggi", xlabel="Juta rupiah")
    for ax in axes[:2]:
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=.2)
    axes[2].grid(axis="x", alpha=.2)
    fig.tight_layout()
    plt.show()
print(f"Insight: kategori omzet terbesar adalah {category_sales.iloc[0].category} "
      f"({category_sales.iloc[0].revenue_share:.1%} dari omzet). "
      "Periksa perubahan tren mingguan bersama konteks stok dan kalender toko.")


## 3. Pola permintaan

Bandingkan hari dalam minggu dan hitung laju jual 28 hari per produk. Laju jual adalah **penjualan tercatat**, bukan permintaan murni: kita tidak punya riwayat stok harian untuk mengetahui kapan barang tidak tersedia. Produk tanpa transaksi 28 hari ditandai untuk ditinjau, bukan langsung dianggap tidak diminati.


In [ ]:
weekday_names = ["Sen", "Sel", "Rab", "Kam", "Jum", "Sab", "Min"]
calendar_days = pd.date_range(transactions.transaction_date.min(), transactions.transaction_date.max(), freq="D")
days_per_weekday = pd.Series(calendar_days.weekday).value_counts().reindex(range(7), fill_value=0)
weekday_sales = sales.groupby(sales.transaction_date.dt.weekday).agg(
    units_sold=("quantity", "sum"), revenue=("revenue", "sum")
).reindex(range(7), fill_value=0)
weekday_sales["days_observed"] = days_per_weekday.to_numpy()
weekday_sales["avg_units_per_day"] = weekday_sales.units_sold / weekday_sales.days_observed
weekday_sales.index = weekday_names
print(weekday_sales.round(1).to_string())

recent_start = snapshot - pd.Timedelta(days=28)
recent_sales = sales.loc[
    (sales.transaction_date >= recent_start) & (sales.transaction_date < snapshot)
].groupby("product_id").agg(
    units_sold_28d=("quantity", "sum"), revenue_28d=("revenue", "sum")
)
print(f"\nPeriode terbaru: {recent_start.date()} s.d. {(snapshot - pd.Timedelta(days=1)).date()}")
print(f"Produk dengan penjualan tercatat: {len(recent_sales):,}")


In [ ]:
# Grafik 4–6: pola hari, sebaran penjualan SKU, dan kategori terbaru.
stocked_recent = products.loc[products.stock_on_hand > 0, ["product_id", "category"]].copy()
stocked_recent = stocked_recent.join(recent_sales[["units_sold_28d"]], on="product_id")
stocked_recent["units_sold_28d"] = stocked_recent.units_sold_28d.fillna(0)
recent_bins = pd.cut(
    stocked_recent.units_sold_28d, bins=[-1, 0, 1, 3, 7, np.inf],
    labels=["0", "1", "2–3", "4–7", "8+"]
).value_counts().reindex(["0", "1", "2–3", "4–7", "8+"], fill_value=0)
recent_category = sales.loc[
    (sales.transaction_date >= recent_start) & (sales.transaction_date < snapshot)
].groupby("category").quantity.sum().sort_values(ascending=False).head(10).sort_values()
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    axes[0].bar(weekday_sales.index, weekday_sales.avg_units_per_day, color="#367c8d")
    axes[0].set(title="Rata-rata unit terjual per hari", ylabel="Unit per hari")
    axes[1].bar(recent_bins.index.astype(str), recent_bins.values, color="#e0a13f")
    axes[1].set(title="Produk berstok menurut unit terjual 28 hari", xlabel="Unit terjual per produk", ylabel="Jumlah produk")
    axes[2].barh(recent_category.index, recent_category.values, color="#74a892")
    axes[2].set(title="Kategori terlaris dalam 28 hari", xlabel="Unit")
    for ax in axes:
        ax.grid(axis="y" if ax is not axes[2] else "x", alpha=.2)
    fig.tight_layout()
    plt.show()
peak_day = weekday_sales.avg_units_per_day.idxmax()
print(f"Insight: {peak_day} mencatat rata-rata unit per hari tertinggi. "
      f"Sebanyak {recent_bins.loc['0']:,} produk berstok tidak mencatat penjualan 28 hari terakhir; "
      "cek ketersediaan dan penempatan produk sebelum menyebutnya tidak diminati.")


## 4. Harga, diskon, dan margin

Ringkas apa yang *tercatat* pada tiap rentang diskon. Perbedaan penjualan antar-rentang adalah **asosiasi**, bukan bukti kenaikan akibat diskon: produk, hari, event, dan ketersediaan stok dapat berbeda. Margin kotor di sini memakai `unit_cost` katalog saat ini sebagai pendekatan; riwayat biaya per transaksi belum tersedia.


In [ ]:
discount_bins = [-0.001, 0.000001, 0.05, 0.10, 0.20, 1.01]
discount_labels = ["0%", ">0–5%", ">5–10%", ">10–20%", ">20%"]
sales["discount_band"] = pd.cut(sales.discount_rate, bins=discount_bins, labels=discount_labels)
sales["gross_profit_proxy"] = sales.quantity * (sales.unit_sale_price - sales.unit_cost)
discount_summary = sales.groupby("discount_band", observed=True).agg(
    units_sold=("quantity", "sum"), revenue=("revenue", "sum"),
    avg_sale_price=("unit_sale_price", "mean"), gross_profit_proxy=("gross_profit_proxy", "sum")
)
discount_summary["gross_margin_proxy"] = discount_summary.gross_profit_proxy / discount_summary.revenue
print(discount_summary.round({"avg_sale_price": 0, "gross_margin_proxy": 3}).to_string())
print("\nCatatan: band diskon bisa memiliki campuran produk dan periode yang berbeda.")


In [ ]:
# Grafik 7–9: campuran tingkat diskon, margin historis, dan jenis promosi.
promo_counts = sales.promotion_type.value_counts().sort_values()
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    axes[0].bar(discount_summary.index.astype(str), discount_summary.units_sold / discount_summary.units_sold.sum() * 100,
                color="#367c8d")
    axes[0].set(title="Porsi unit terjual per tingkat diskon", ylabel="Persen dari unit terjual")
    axes[1].bar(discount_summary.index.astype(str), discount_summary.gross_margin_proxy * 100,
                color="#e07a3f")
    axes[1].axhline(0, color="#555555", linewidth=.8)
    axes[1].set(title="Margin kotor pendekatan", ylabel="Persen dari omzet")
    axes[2].barh(promo_counts.index, promo_counts.values, color="#74a892")
    axes[2].set(title="Jumlah transaksi per jenis promosi", xlabel="Transaksi")
    for ax in axes[:2]:
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=.2)
    axes[2].grid(axis="x", alpha=.2)
    fig.tight_layout()
    plt.show()
print("Insight: margin per band diskon terlihat berbeda, tetapi campuran produk dan waktu juga berbeda. "
      "Grafik ini belum mengukur tambahan penjualan akibat diskon.")


## 5. Risiko stok pada tanggal snapshot

Untuk setiap produk dengan stok positif, hitung `days_to_expiry`, penjualan 28 hari, dan *days of cover* (`stock_on_hand / rata-rata unit terjual per hari`). Untuk produk yang kedaluwarsa dalam **90 hari**, `units_at_risk_proxy` adalah stok dikurangi perkiraan unit yang terjual sampai tanggal kedaluwarsa **jika laju 28 hari berlanjut**. Di luar jendela itu indikator tidak dihitung karena ekstrapolasinya terlalu jauh. Ini indikator penyaringan, bukan forecast atau angka kerugian pasti. Produk yang sudah mencapai tanggal kedaluwarsa dipisahkan untuk pemeriksaan kelayakan jual, bukan dipromosikan.


In [ ]:
inventory = products.loc[products.stock_on_hand > 0, [
    "product_id", "sku", "product_name", "category", "unit_cost", "list_price", "stock_on_hand", "expiry_date"
]].copy()
inventory = inventory.join(recent_sales, on="product_id")
inventory[["units_sold_28d", "revenue_28d"]] = inventory[["units_sold_28d", "revenue_28d"]].fillna(0)
inventory["units_sold_28d"] = inventory.units_sold_28d.astype(int)
inventory["days_to_expiry"] = (inventory.expiry_date - snapshot).dt.days
inventory["daily_sales_rate_28d"] = inventory.units_sold_28d / 28
inventory["days_of_cover"] = inventory.stock_on_hand / inventory.daily_sales_rate_28d.replace(0, np.nan)
inventory["expected_units_until_expiry_proxy"] = (
    inventory.daily_sales_rate_28d * inventory.days_to_expiry.clip(lower=0)
).clip(upper=inventory.stock_on_hand)
inventory["units_at_risk_proxy"] = (
    inventory.stock_on_hand - inventory.expected_units_until_expiry_proxy
).clip(lower=0).round().astype("Int64")
inventory.loc[inventory.days_to_expiry > 90, "units_at_risk_proxy"] = pd.NA
inventory["cost_at_risk_proxy"] = inventory.units_at_risk_proxy * inventory.unit_cost
inventory["gross_margin_rate_at_list_price"] = (inventory.list_price - inventory.unit_cost) / inventory.list_price

conditions = [
    inventory.days_to_expiry <= 0,
    inventory.days_to_expiry <= 30,
    inventory.days_to_expiry <= 90,
    inventory.units_sold_28d == 0,
]
choices = ["check_expiry", "expiry_30d", "expiry_90d", "no_recent_sales"]
inventory["priority_group"] = np.select(conditions, choices, default="routine")
summary = inventory.groupby("priority_group").agg(
    products=("product_id", "count"), stock_units=("stock_on_hand", "sum"),
    units_at_risk_proxy=("units_at_risk_proxy", lambda values: values.sum(min_count=1)),
    cost_at_risk_proxy=("cost_at_risk_proxy", lambda values: values.sum(min_count=1))
).reindex(["check_expiry", "expiry_30d", "expiry_90d", "no_recent_sales", "routine"], fill_value=0)
print(summary.to_string())


In [ ]:
# Grafik 10–12: komposisi stok, modal yang berisiko, dan stok versus laju jual.
expiry_order = ["check_expiry", "expiry_30d", "expiry_90d", "no_recent_sales", "routine"]
expiry_labels = ["Cek tanggal", "≤30 hari", "31–90 hari", "Tanpa jual 28h", "Rutin"]
risk_by_category = inventory.loc[inventory.days_to_expiry <= 90].groupby("category").cost_at_risk_proxy.sum()
risk_by_category = risk_by_category.nlargest(10).sort_values()
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    bars = axes[0].bar(expiry_labels, summary.loc[expiry_order, "stock_units"] / 1_000,
                       color=["#b54c4c", "#e07a3f", "#e0a13f", "#7da7b5", "#74a892"])
    axes[0].set(title="Unit stok menurut kelompok prioritas", ylabel="Ribu unit")
    axes[0].tick_params(axis="x", rotation=25)
    axes[1].barh(risk_by_category.index, risk_by_category.values / 1_000_000, color="#b54c4c")
    axes[1].set(title="10 kategori: modal berisiko (proksi)", xlabel="Juta rupiah")
    scatter = inventory.loc[inventory.days_to_expiry > 0].sample(
        n=min(2000, (inventory.days_to_expiry > 0).sum()), random_state=42
    )
    urgent = scatter.days_to_expiry.le(30)
    axes[2].scatter(scatter.loc[~urgent, "units_sold_28d"] + 1, scatter.loc[~urgent, "stock_on_hand"] + 1,
                    alpha=.35, s=12, color="#367c8d", label=">30 hari")
    axes[2].scatter(scatter.loc[urgent, "units_sold_28d"] + 1, scatter.loc[urgent, "stock_on_hand"] + 1,
                    alpha=.65, s=15, color="#e07a3f", label="≤30 hari")
    axes[2].set(xscale="log", yscale="log", title="Stok vs penjualan 28 hari",
                xlabel="Unit terjual 28 hari + 1 (log)", ylabel="Stok unit + 1 (log)")
    axes[2].legend(frameon=False)
    for ax in axes[:2]: ax.grid(axis="y" if ax is axes[0] else "x", alpha=.2)
    fig.tight_layout()
    plt.show()
near_expiry = inventory.days_to_expiry.between(1, 30)
print(f"Insight: {near_expiry.sum():,} produk dengan {inventory.loc[near_expiry, 'stock_on_hand'].sum():,} unit "
      "stok berada dalam 30 hari menuju kedaluwarsa. Nilai risiko adalah proksi, bukan kerugian pasti.")


## 6. Daftar prioritas produk

Urutkan produk yang perlu pemeriksaan tanggal kedaluwarsa, lalu stok yang berpotensi tersisa dalam 30 dan 90 hari. Nilai risiko hanyalah pendekatan untuk *triage*. Tim toko tetap perlu memeriksa stok fisik, batch kedaluwarsa, dan kelayakan jual sebelum bertindak. Notebook **tidak** mengeluarkan besaran diskon karena hubungan kausal harga–permintaan belum diketahui.


In [ ]:
priority_order = {"check_expiry": 0, "expiry_30d": 1, "expiry_90d": 2, "no_recent_sales": 3, "routine": 4}
priority_products = inventory.assign(
    priority_rank=inventory.priority_group.map(priority_order)
).sort_values(
    ["priority_rank", "cost_at_risk_proxy", "days_to_expiry"],
    ascending=[True, False, True]
)[[
    "product_id", "sku", "product_name", "category", "priority_group", "stock_on_hand",
    "units_sold_28d", "days_of_cover", "days_to_expiry", "units_at_risk_proxy",
    "cost_at_risk_proxy", "gross_margin_rate_at_list_price"
]].reset_index(drop=True)
preview = priority_products.head(20).copy()
preview["days_of_cover"] = preview.days_of_cover.map(lambda x: "—" if pd.isna(x) else f"{x:,.0f}")
print(preview.to_string(index=False, formatters={
    "gross_margin_rate_at_list_price": "{:.1%}".format,
}))

# Jika perlu file untuk dashboard/peninjauan manual, ubah menjadi True lalu jalankan sel ini.
EXPORT_PRIORITY_CSV = False
if EXPORT_PRIORITY_CSV:
    output_path = Path.cwd() / "priority_products.csv"
    priority_products.to_csv(output_path, index=False)
    print("File tersimpan:", output_path)


In [ ]:
# Grafik 13–14: jumlah produk dan stok dalam daftar prioritas.
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].bar(expiry_labels, summary.loc[expiry_order, "products"],
                color=["#b54c4c", "#e07a3f", "#e0a13f", "#7da7b5", "#74a892"])
    axes[0].set(title="Jumlah produk per kelompok prioritas", ylabel="Produk")
    axes[0].tick_params(axis="x", rotation=25)
    urgent_products = priority_products.loc[priority_products.priority_group == "expiry_30d"].head(10)
    axes[1].barh(urgent_products.product_id.iloc[::-1],
                 urgent_products.cost_at_risk_proxy.iloc[::-1] / 1_000_000, color="#e07a3f")
    axes[1].set(title="10 produk ≤30 hari: modal berisiko (proksi)", xlabel="Juta rupiah")
    for ax in axes: ax.grid(axis="y" if ax is axes[0] else "x", alpha=.2)
    fig.tight_layout()
    plt.show()
print("Insight: mulai peninjauan dari kelompok 'check_expiry', lalu produk ≤30 hari dengan stok dan "
      "modal berisiko terbesar. Verifikasi tanggal batch dan kondisi fisik sebelum tindakan harga.")


## Kesimpulan dan langkah berikutnya

Baca `quality_report`, `weekly_sales`, `category_sales`, `weekday_sales`, `discount_summary`, dan `priority_products` sebagai enam output EDA. Pilih beberapa produk prioritas dan cocokkan dengan kondisi toko. Angka *days of cover* yang ekstrem atau stok berisiko besar pada data prototipe adalah sinyal untuk memeriksa kewajaran data, bukan alasan otomatis memberi diskon.

Agar Kapanin bisa merekomendasikan **waktu dan besaran diskon** dengan lebih kredibel, kumpulkan riwayat stok harian/batch, penerimaan dan pembuangan stok, perubahan biaya, serta jadwal promosi dan produk yang tidak ikut promosi. Setelah itu tetapkan target bisnis (misalnya laba kotor setelah potensi barang kedaluwarsa) dan uji beberapa tingkat diskon secara terkontrol. Barulah model efek diskon dan evaluasi kebijakan layak dibuat.
